# Lektion 11 - Agent-til-Agent (A2A) protokol


## Opsætning


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hvad er A2A-protokollen?

Den **Agent-to-Agent (A2A) protokol** er en åben standard, der gør det muligt for AI-agenter at kommunikere,
finde hinanden og samarbejde — selv når de er bygget på forskellige rammer eller hostet
af forskellige tjenester.

Nøglebegreber:

- **Opdagelse** – Agenter offentliggør et *Agentkort*, der beskriver deres kapaciteter, hvilket gør det
  nemt for andre agenter (eller orkestratorer) at finde den rette specialist til en opgave.
- **Beskedudveksling** – Agenter udveksler strukturerede beskeder gennem en fælles protokol, så en
  forespørgsel fra en agent kan forstås og opfyldes af en anden uanset dens interne
  implementering.
- **Opgavens livscyklus** – A2A definerer tilstande som *submitted*, *working*, *completed*, og
  *failed*, hvilket giver orkestratoren fuld indsigt i, hvordan en delegeret opgave skrider frem.

I denne lektion simulerer vi A2A-lignende samarbejde ved at sammenkoble tre specialiserede rejseagenter
i en arbejdsgang, hvor hver agent bidrager med sin ekspertise og videresender resultater til den næste.


## Oprettelse af specialiserede rejseagenter


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Samarbejde mellem flere agenter via arbejdsgang

Vi kobler de tre agenter sammen i en sekventiel arbejdsgang, der afspejler A2A-beskedudveksling:

1. **CurrencyExchangeAgent** modtager brugerens forespørgsel og giver vejledning om valuta.
2. **ActivityPlannerAgent** modtager den berigede kontekst og tilføjer anbefalinger til aktiviteter.
3. **TravelManagerAgent** sammenstiller begge input til en endelig rejseoversigt.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Forståelse af A2A i produktion

I et produktionsmiljø åbner A2A-protokollen for kraftfulde tværgående service-scenarier:

| Funktionalitet | Beskrivelse |
|---|---|
| **Interoperabilitet på tværs af frameworks** | En agent bygget med ét framework kan delegere opgaver til en agent bygget med ethvert andet A2A-kompatibelt framework, hvilket muliggør ægte interoperabilitet på tværs af organisationer. |
| **Servicegrænser** | Agenter kan være placeret i separate mikrotjenester, cloud-regioner eller endda forskellige organisationer, mens de stadig samarbejder problemfrit. |
| **Dynamisk opdagelse** | En orkestrator kan forespørge et Agent Card-register i runtime for at finde den bedst egnede specialist til en given underopgave. |
| **Streaming & push-notifikationer** | A2A understøtter Server-Sent Events (SSE) for realtidsopdateringer om fremdrift og push-notifikationer til langvarige opgaver. |

Den arbejdsgang, vi byggede ovenfor, er en forenklet, in-process-version af dette mønster. I en reel udrulning ville hver agent udstille et HTTP-endpoint, publicere et Agent Card og kommunikere via A2A JSON-RPC-protokollen.


## Resumé

I denne lektion lærte du:

1. **Hvad A2A-protokollen er** — en åben standard for agent-til-agent opdagelse, beskeder,
   og opgavestyring.
2. **Hvordan man opretter specialiserede agenter** — en Currency Exchange agent, en Activity Planner agent,
   og en Travel Manager orchestrator.
3. **Hvordan man sammenkobler agenter i en workflow** — ved at bruge `WorkflowBuilder` til at modellere sekventiel
   meddelelsesudveksling mellem agenter.
4. **Hvordan A2A fungerer i produktion** — muliggør samarbejde på tværs af frameworks og services
   med dynamisk opdagelse og streamingopdateringer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Ansvarsfraskrivelse:
Dette dokument er oversat ved hjælp af AI-oversættelsestjenesten Co-op Translator (https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiske oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på originalsproget skal betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi er ikke ansvarlige for eventuelle misforståelser eller fejltolkninger som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
